# DFX4ML Multi-Testcase Sweep Notebook

Runs the same DFX4ML end-to-end pipeline for multiple `AMT_QUERY` sizes in a single session.

## Testcase Sizes

| # | AMT_QUERY |
|---|---|
| 1 | 100 |
| 2 | 500 |
| 3 | 1000 |
| 4 | 3000 |
| 5 | 5000 |
| 6 | 7000 |
| 7 | 9000 |
| 8 | 11000 |
| 9 | 13000 |
| 10 | 15000 |
| 11 | 17000 |

Hardware setup (overlay, DFX controller, partial bitstreams) is performed **once** before the loop.
Each testcase re-initialises the DFX Manager and allocates fresh DMA buffers.

> **Hardware Setup Required:** This notebook must run on a PYNQ board with the correct bitstreams placed in `hw/`.

In [ ]:
# import the library
## pynq
from pynq import Overlay
from pynq import allocate
from pynq import DefaultIP
from pynq import Interrupt
## standard python
import asyncio
import os
import time
import numpy as np
## our driver
import driver.cap           as cap
import driver.mem_alloc     as dataAlloc
import driver.dfx_unified   as dfx_unified
import driver.dfx_mgs_debug as dfx_mgs_debug

## Step 1 — Project Configuration

| Variable | Description |
|---|---|
| `FULL_BS_NAME` | Static (full) bitstream |
| `PAR_BS_NAME_0/1` | Partial bitstreams for RM slot 0 and 1 |
| `QUERY_SIZES` | List of `AMT_QUERY` values to sweep |
| `AMT_SLOT` | Number of reconfigurable slots |
| `INPUT_DATA_NAME` | Source `.npy` file (must have ≥ max(QUERY_SIZES) rows) |

In [ ]:
# path parameter
PRJ_DIR         = os.getcwd()
PRJ_HW_DIR      = PRJ_DIR + '/hw/'
PRJ_TC_DIR      = PRJ_DIR + '/data/'
DFX_CONFIG_FILE = 'dfx_ctrl_con.txt'
# reconfigurable module parameter
AMT_SLOT        = 2
FULL_BS_NAME    = 'system.bin'
PAR_BS_NAME_0   = 'rm_0.bin'
PAR_BS_NAME_1   = 'rm_2.bin'
# input data parameter
INPUT_DATA_NAME = 'x_input_20000.npy'
# multi-testcase query sizes
QUERY_SIZES     = [100, 500, 1000, 3000, 5000, 7000, 9000, 11000, 13000, 15000, 17000]

print(f"Query sizes to sweep: {QUERY_SIZES}")

## Step 2 — Load the Full Bitstream (Static Overlay)

Switch PL configuration to **PCAP**, then load the full bitstream via PYNQ's `Overlay`.

In [ ]:
cap.change_pl_config_mode("pcap", True, "")

In [ ]:
overlay = Overlay(PRJ_HW_DIR + FULL_BS_NAME)

## Step 3 — Set Up Interrupt

Register the interrupt pin from `dfx_unified_0` for asynchronous completion signalling.

In [ ]:
overlay.interrupt_pins

In [ ]:
my_interrupt = Interrupt('dfx_unified_0/dfx_intr')

## Step 4 — Access IP Blocks

| Python Handle | Hardware IP | Role |
|---|---|---|
| `dma_ip` | AXI DMA | Moves data between DDR and reconfigurable region |
| `dfx_ctrl` | DFX Controller | Triggers partial reconfiguration via ICAP |
| `dfx_mng` | DFX Manager | Orchestrates DMA→compute→reconfig sequence |

In [ ]:
dfx_unifed_ip = overlay.dfx_unified_0

In [ ]:
dma_ip   = dfx_unifed_ip.dfx_dma
dfx_ctrl = dfx_unifed_ip.dfx_ctrl
dfx_mng  = dfx_unifed_ip.dfx_mng
pr_ctrl  = dfx_unifed_ip.pr_ctrl

## Step 5 — Configure the DFX Controller

Load controller metadata from `dfx_ctrl_con.txt`, then switch PL interface from **PCAP → ICAP**.

In [ ]:
dfx_ctrl.config(PRJ_HW_DIR + DFX_CONFIG_FILE)
print("regIdxSize = ", dfx_ctrl.BLS_REGID)

In [ ]:
cap.change_pl_config_mode("icap", True, "")

## Step 6 — Initial System Reset

Soft-reset both engines to a clean idle state.

In [ ]:
dfx_ctrl.print_status()

In [ ]:
dfx_mng .shutdown_engine()
dfx_ctrl.shutdown_engine()

## Step 7 — Load Partial Bitstreams into CMA Memory

Allocate CMA buffers for each partial bitstream and register their addresses in the DFX Controller.
This is done **once** and shared across all testcases.

In [ ]:
print("------ allocate bitstream CMA for each trigger ------")

d0_ip_buf, d0_addr, d0_size = \
    dfx_ctrl.allocate_bit_stream_cma(PRJ_HW_DIR + PAR_BS_NAME_0)

d1_ip_buf, d1_addr, d1_size = \
    dfx_ctrl.allocate_bit_stream_cma(PRJ_HW_DIR + PAR_BS_NAME_1)

In [ ]:
dfx_ctrl.set_simple_meta_data(0, d0_addr, d0_size)
dfx_ctrl.set_simple_meta_data(1, d1_addr, d1_size)

In [ ]:
dfx_ctrl.print_status()
dfx_ctrl.print_simple_meta_data(0)
dfx_ctrl.print_simple_meta_data(1)

## Step 8 — Physical Addresses

Capture DMA and DFX Controller physical addresses used when programming the DFX Manager.

In [ ]:
dmaPhyAddr     = 0xA003_0000
prPhyAddr      = 0xA005_0000
dfxCtrlPhyAddr = 0

print("dma physical address:      ", hex(dmaPhyAddr))
print("dfx ctrl physical address: ", hex(dfxCtrlPhyAddr))

## Step 9 — Pre-load Full Input Data

Load the `.npy` file once; each testcase will slice the required number of rows from it.

In [ ]:
inputX_full = np.load(PRJ_TC_DIR + INPUT_DATA_NAME)
print(f"Full input loaded: {inputX_full.shape}  (max available rows = {inputX_full.shape[0]})")
assert inputX_full.shape[0] >= max(QUERY_SIZES), \
    f"Input file has only {inputX_full.shape[0]} rows but max QUERY_SIZES={max(QUERY_SIZES)}"

## Step 10 — Define Async Execution Helper

Reusable coroutine that starts the DFX Manager engine and waits for the hardware interrupt.

In [ ]:
async def startExecAndWait4Intr():
    start_time = time.perf_counter()
    dfx_mng.clear_intr()
    dfx_mng.start_engine()
    while True:
        await my_interrupt.wait()
        end_time = time.perf_counter()
        elapsed  = end_time - start_time
        print(f"  interrupt received  |  elapsed: {elapsed:.6f} s")
        break
    return elapsed

In [ ]:
loop2 = asyncio.get_event_loop()

## Step 11 — Multi-Testcase Execution Loop

For each `AMT_QUERY` in `QUERY_SIZES`:
1. Reset engines
2. Slice input data to `AMT_QUERY` rows
3. Allocate DMA I/O buffers
4. Initialise DFX Manager slot table
5. Trigger RM_0 initial load
6. Execute pipeline and wait for interrupt
7. Read back output and save to `data/output_result_q{AMT_QUERY}.npy`

A timing summary is printed at the end.

In [ ]:
results_summary = []

for AMT_QUERY in QUERY_SIZES:
    print(f"\n{'='*60}")
    print(f"  Testcase  AMT_QUERY = {AMT_QUERY}")
    print(f"{'='*60}")

    INPUT_SHAPE  = (AMT_QUERY, 8, 8, 1)
    OUTPUT_SHAPE = (AMT_QUERY, 4)

    # ---- reset engines ----
    dfx_mng .shutdown_engine()
    dfx_ctrl.shutdown_engine()

    # ---- slice input ----
    inputX = inputX_full[:AMT_QUERY]
    expected_spatial = INPUT_SHAPE[1:]   # (8, 8, 1)
    if inputX.shape[1:] != expected_spatial:
        print(f"  WARNING: spatial shape {inputX.shape[1:]} != {expected_spatial}, slicing")
        idx = tuple([slice(None)] + [slice(s) for s in expected_spatial])
        inputX = inputX[idx]
    assert inputX.shape == INPUT_SHAPE, f"Shape mismatch: {inputX.shape} != {INPUT_SHAPE}"
    print(f"  Input slice : {inputX.shape}")

    # ---- allocate DMA buffers ----
    buf_input, buf_input_phya, buf_input_sz = dataAlloc.alloc_data_uint(
        alloc_shape=INPUT_SHAPE,  alloc_type=np.float32, input_x=inputX)
    buf_out,   buf_out_phy,   buf_out_sz   = dataAlloc.alloc_data_uint(
        alloc_shape=OUTPUT_SHAPE, alloc_type=np.float32)
    buf_input.flush()

    # ---- init DFX manager ----
    dfx_mng.set_end_cnt(AMT_SLOT - 1)
    dfx_mng.set_dma_addr(dmaPhyAddr)
    dfx_mng.set_dfx_addr(dfxCtrlPhyAddr)
    dfx_mng.set_pr_ctrl_addr(prPhyAddr)
    dfx_mng.set_batch_size(AMT_QUERY)
    dfx_mng.set_intr_ena(1)
    dfx_mng.set_intr(1)
    dfx_mng.set_round_trip(0)

    #                         [srcPhyAddr    ,        srcSz,  dstPhyAddr,      dstSz,st,pr,loadMask, storeMask, intrMask, profile_exec]
    dfx_mng.set_whole_slot(0, [buf_input_phya, buf_input_sz,           0,          0, 0, 0,  0b0001, 0b0010, 0, 0])
    dfx_mng.set_whole_slot(1, [             0,            0, buf_out_phy, buf_out_sz, 0, 0,  0b0010, 0b0001, 0, 0])

    # ---- initial DFX trigger: load RM_0 ----
    dfx_unifed_ip.dfx_man.grant_decoupler_to_dfx_ctrl()
    dfx_ctrl.trig(0)
    dfx_ctrl.restart_no_status()

    # ---- execute pipeline ----
    task = loop2.create_task(startExecAndWait4Intr())
    loop2.run_until_complete(task)
    elapsed = task.result()

    dfx_mng.shutdown_engine()

    # ---- profile counters per slot ----
    # get_slot() returns: (srcAddr, srcSz, dstAddr, dstSz, status,
    #                      profileCnt, ldMask, stMask, intrMask, profileExecCnt)
    print(f"  {'slot':>6}  {'profileCnt':>12}  {'profileExecCnt':>16}")
    print(f"  {'------':>6}  {'----------':>12}  {'---------------':>16}")
    slot_profiles = []
    for s in range(AMT_SLOT):
        slot_data    = dfx_mng.get_slot(s)
        profile_cnt  = slot_data[5]
        profile_exec = slot_data[9]
        print(f"  {s:>6d}  {profile_cnt:>12}  {profile_exec:>16}")
        slot_profiles.append({"profileCnt": profile_cnt, "profileExecCnt": profile_exec})

    # ---- read back results ----
    buf_out.invalidate()
    np_parRes = np.array(buf_out, dtype=np.float32).reshape(OUTPUT_SHAPE)
    out_fname = f"output_result_q{AMT_QUERY}.npy"
    np.save(os.path.join(PRJ_TC_DIR, out_fname), np_parRes)
    print(f"  Saved : {out_fname}  shape={np_parRes.shape}")

    results_summary.append({"amt_query": AMT_QUERY, "elapsed_s": elapsed, "slots": slot_profiles})

# ---- print summary table ----
SEP = "="*72
HDR = f"  {'AMT_QUERY':>10}  {'elapsed (s)':>14}  {'slot':>6}  {'profileCnt':>12}  {'profileExecCnt':>16}"
DIV = f"  {'-'*10}  {'-'*14}  {'------':>6}  {'----------':>12}  {'---------------':>16}"

summary_lines = [SEP, "  SWEEP SUMMARY", SEP, HDR, DIV]
for r in results_summary:
    for s, sp in enumerate(r["slots"]):
        q_col = f"{r['amt_query']:>10d}" if s == 0 else f"{'':>10}"
        t_col = f"{r['elapsed_s']:>14.6f}" if s == 0 else f"{'':>14}"
        summary_lines.append(
            f"  {q_col}  {t_col}  {s:>6d}  {sp['profileCnt']:>12}  {sp['profileExecCnt']:>16}"
        )
summary_lines.append(SEP)

print("\n\n" + "\n".join(summary_lines))

# ---- save summary to files ----
summary_txt_path = os.path.join(PRJ_TC_DIR, "sweep_summary.txt")
with open(summary_txt_path, "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print(f"\nSaved text summary -> {summary_txt_path}")

summary_csv_path = os.path.join(PRJ_TC_DIR, "sweep_summary.csv")
with open(summary_csv_path, "w") as f:
    f.write("amt_query,elapsed_s,slot,profileCnt,profileExecCnt\n")
    for r in results_summary:
        for s, sp in enumerate(r["slots"]):
            f.write(f"{r['amt_query']},{r['elapsed_s']:.9f},{s},{sp['profileCnt']},{sp['profileExecCnt']}\n")
print(f"Saved CSV  summary -> {summary_csv_path}")

## Step 12 — Debug: Read DFX Manager State (Optional)

Uncomment to inspect internal MGS registers after the sweep.

In [ ]:
# mgs_dbg = dfx_mgs_debug.DFX_mgs_debug(overlay.axi_gpio_0)
# mgs_dbg.read_mgs_meta()